# 02 · Supervised fine-tuning for question answering (Colab, T4 GPU)

Continues training the **pretrained** model on question–answer pairs:
`<|user|> question<|assistant|> answer<|endoftext|>`. The loss counts only the answer tokens.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
Needs the pretrained checkpoint at `MyDrive/mini-gpt/pretrain_run2/ckpt_best.pt` (from notebook 01).
Output goes to `MyDrive/mini-gpt/sft/`. Re-running after a disconnect **resumes** automatically.

📸 = take a screenshot of that cell's output for the report.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/mini-gpt'
PRETRAINED = f'{DRIVE}/pretrain_run2/ckpt_best.pt'
RUN_DIR = f'{DRIVE}/sft'
RESULTS = f'{DRIVE}/results'
assert os.path.exists(PRETRAINED), f'pretrained checkpoint not found: {PRETRAINED}'
os.makedirs(RUN_DIR, exist_ok=True)
for sub in ('figures', 'logs', 'samples'):
    os.makedirs(f'{RESULTS}/{sub}', exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO = 'https://github.com/Shayan047/mini-gpt.git'
if os.path.exists('/content/mini-gpt'):
    !git -C /content/mini-gpt pull
else:
    !git clone {REPO} /content/mini-gpt
%cd /content/mini-gpt
!git log --oneline -1
!python -m tests.test_model && python -m tests.test_finetune

In [ ]:
# 📸 Fine-tuning data summary (report: fine-tuning)
print(open('results/tables/sft_data_stats.md').read())

## Fine-tune
1,000 steps × batch 32 ≈ 2.9 epochs over 11,056 pairs, AdamW lr 1e-4 (warmup 50, cosine to 1e-5), fp16.
Step 0 is logged before any update, so the log starts with the pretrained model's numbers. Expect a few minutes on a T4.

In [ ]:
# 📸 The full output of this cell (fine-tuning log)
!python -m src.finetune --init "{PRETRAINED}" --out-dir "{RUN_DIR}"

In [ ]:
# 📸 Fine-tuning curves
!python -m src.plot_sft --log "{RUN_DIR}/log.csv" --out "{RESULTS}/figures/sft_loss.png"
from IPython.display import Image, display
display(Image(f'{RESULTS}/figures/sft_loss.png'))

In [ ]:
# How answers to fixed validation questions changed during fine-tuning
import json
samples = [json.loads(l) for l in open(f'{RUN_DIR}/samples.jsonl')]
steps = sorted({s['step'] for s in samples})
for step in (steps[0], steps[1], steps[-1]):
    print(f'================ step {step} ================')
    for s in samples:
        if s['step'] == step:
            print('Q:', s['question']); print('A:', s['answer'][:200]); print()

In [ ]:
# Copy logs to the Drive results folder (download into the repo's results/ folder)
import shutil
for f in ('log.csv', 'train_log.txt', 'summary.json'):
    shutil.copy(f'{RUN_DIR}/{f}', f'{RESULTS}/logs/sft_{f}')
shutil.copy(f'{RUN_DIR}/samples.jsonl', f'{RESULTS}/samples/sft_samples_during_training.jsonl')
print(open(f'{RUN_DIR}/summary.json').read())
print('Download', f'{RUN_DIR}/ckpt_best.pt', '-> repo checkpoints/sft/ckpt_best.pt')